# 03 — GNN Training

Trains the Graph Convolutional Network on the worker graph. Requires PyTorch and PyTorch Geometric.

Install:
```
pip install torch torch_geometric scikit-learn shap
```

With only 8 nodes in the synthetic dataset the model runs in seconds. Once you connect real CMIE CPHS data (hundreds of household clusters), training becomes meaningful.

In [ ]:
import sys; sys.path.insert(0, '..')
import torch
import pandas as pd
import matplotlib.pyplot as plt

from src.models.train import train, build_pyg_data
from src.graph.builder import load_data

print('PyTorch version:', torch.__version__)

In [ ]:
# Inspect the PyG Data object before training
workers, platforms, skills, districts, edges = load_data(use_synthetic=True)
data, worker_ids = build_pyg_data(workers, edges)

print('Node feature matrix shape:', data.x.shape)
print('Edge index shape:          ', data.edge_index.shape)
print('Labels shape:              ', data.y.shape)
print('Train nodes:               ', data.train_mask.sum().item())
print('Val nodes:                 ', data.val_mask.sum().item())
print('Test nodes:                ', data.test_mask.sum().item())

In [ ]:
# Train the model
# With 8 nodes this is instant. Increase GNN_EPOCHS in config.py for real data.
model, risk_scores = train(use_synthetic=True)
print()
print(risk_scores.to_string(index=False))

In [ ]:
# Merge risk scores back to worker labels and plot
plot_df = workers.merge(risk_scores, on='worker_id', how='left')
plot_df = plot_df.sort_values('risk_score', ascending=True)

colors = ['#e74c3c' if s >= 0.70 else '#f39c12' if s >= 0.45 else '#27ae60'
          for s in plot_df['risk_score']]

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(plot_df['label'], plot_df['risk_score'], color=colors)
ax.axvline(0.70, color='red', linestyle='--', linewidth=0.8, label='High risk threshold')
ax.axvline(0.45, color='orange', linestyle='--', linewidth=0.8, label='Medium risk threshold')
ax.set_xlabel('GNN risk score')
ax.set_title('Worker cluster displacement vulnerability')
ax.legend(fontsize=8)
ax.set_xlim(0, 1)
plt.tight_layout()
plt.savefig('../logs/gnn_risk_scores.png', bbox_inches='tight')
plt.show()

In [ ]:
# Visualise learned embeddings using PCA
from sklearn.decomposition import PCA

embeddings = model.get_embeddings(data.x, data.edge_index).numpy()
pca = PCA(n_components=2)
coords = pca.fit_transform(embeddings)

fig, ax = plt.subplots(figsize=(7, 5))
for i, wid in enumerate(worker_ids):
    label = workers[workers['worker_id'] == wid]['label'].values[0]
    risk  = risk_scores[risk_scores['worker_id'] == wid]['risk_score'].values[0]
    color = '#e74c3c' if risk >= 0.70 else '#f39c12' if risk >= 0.45 else '#27ae60'
    ax.scatter(coords[i, 0], coords[i, 1], c=color, s=120)
    ax.annotate(label[:18], (coords[i, 0], coords[i, 1]),
                textcoords='offset points', xytext=(6, 3), fontsize=7)

ax.set_title('GCN node embeddings (PCA projection, color = risk)')
ax.set_xlabel('PC 1')
ax.set_ylabel('PC 2')
plt.tight_layout()
plt.savefig('../logs/gnn_embeddings_pca.png', bbox_inches='tight')
plt.show()